In [21]:
import numpy as np
import pandas as pd
import neurokit2 as nk
from pathlib import Path
from tqdm import tqdm
import time

from scipy.signal import resample_poly

In [2]:
FS = 200                     # Sampling frequency (Hz)
WINDOW_SEC = 30              # Sliding window length (seconds)
WINDOW_SAMPLES = FS * WINDOW_SEC
STEP_SEC = 1                 # Step size for 1 Hz output


In [3]:
def resample_signal(signal, fs_in=256, fs_out=200):
    """
    Resample signal using polyphase filtering
    """
    return resample_poly(signal, fs_out, fs_in)

In [4]:
import warnings
warnings.filterwarnings("ignore")


In [22]:
DATA_DIR = Path("../data/raw/Participants")


In [23]:
def get_valid_blocks(participant_num):
    base_blocks = list(range(1, 11))  # 1–10

    if 1 <= participant_num <= 11:
        extra_blocks = [15, 20, 25, 30, 32, 34, 36]
    else:
        extra_blocks = [12, 14, 16, 18, 23, 28, 33]

    return set(base_blocks + extra_blocks)

In [8]:
import re

participant_files = []

for part_dir in sorted(
    DATA_DIR.glob("Part*"),
    key=lambda x: int(x.name.replace("Part", ""))
):
    part_num = int(part_dir.name.replace("Part", ""))

    if part_num > 48:
        continue

    by_block_dir = part_dir / "by_block"
    if not by_block_dir.exists():
        print(f"WARNING: {by_block_dir} missing")
        continue

    valid_blocks = get_valid_blocks(part_num)

    for file in by_block_dir.glob("*_gsr_ppg*.csv"):

        # 🔥 Hämta första talet i filnamnet
        match = re.match(r"(\d+)", file.name)

        if not match:
            print(f"Skipping bad filename: {file.name}")
            continue

        block_num = int(match.group(1))

        if block_num in valid_blocks:
            participant_files.append((part_num, block_num, file))

# sortera korrekt
participant_files = sorted(participant_files, key=lambda x: (x[0], x[1]))

print("Total selected files:", len(participant_files))

Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping bad filename: meta_info_gsr_ppg.csv
Skipping b

In [9]:
# välj vilken participant du vill inspektera
target_participant = 1  # ändra denna (t.ex. 12, 25, etc.)

files = [
    (block_num, file)
    for part_num, block_num, file in participant_files
    if part_num == target_participant
]

files_sorted = sorted(files, key=lambda x: x[0])

print(f"\nPart{target_participant} har {len(files_sorted)} filer:\n")

for block_num, file in files_sorted:
    print(f"Block {block_num:2d} → {file.name}")


Part1 har 17 filer:

Block  1 → 1_gsr_ppg_.csv
Block  2 → 2_gsr_ppg_mathtest.csv
Block  3 → 3_gsr_ppg_.csv
Block  4 → 4_gsr_ppg_.csv
Block  5 → 5_gsr_ppg_strooptest.csv
Block  6 → 6_gsr_ppg_.csv
Block  7 → 7_gsr_ppg_.csv
Block  8 → 8_gsr_ppg_IQtest.csv
Block  9 → 9_gsr_ppg_.csv
Block 10 → 10_gsr_ppg_.csv
Block 15 → 15_gsr_ppg_.csv
Block 20 → 20_gsr_ppg_.csv
Block 25 → 25_gsr_ppg_.csv
Block 30 → 30_gsr_ppg_.csv
Block 32 → 32_gsr_ppg_.csv
Block 34 → 34_gsr_ppg_.csv
Block 36 → 36_gsr_ppg_.csv


In [10]:
from collections import Counter

count_per_participant = Counter([p for p, _, _ in participant_files])

print("\nFiles per participant:")
for p in range(1, 49):
    print(f"Part{p}: {count_per_participant[p]}")


Files per participant:
Part1: 17
Part2: 17
Part3: 17
Part4: 17
Part5: 17
Part6: 17
Part7: 17
Part8: 17
Part9: 17
Part10: 17
Part11: 17
Part12: 17
Part13: 17
Part14: 17
Part15: 17
Part16: 17
Part17: 17
Part18: 17
Part19: 17
Part20: 17
Part21: 17
Part22: 17
Part23: 17
Part24: 17
Part25: 17
Part26: 17
Part27: 17
Part28: 17
Part29: 17
Part30: 17
Part31: 17
Part32: 17
Part33: 17
Part34: 17
Part35: 17
Part36: 17
Part37: 17
Part38: 17
Part39: 17
Part40: 17
Part41: 17
Part42: 17
Part43: 17
Part44: 17
Part45: 17
Part46: 17
Part47: 17
Part48: 17


In [11]:
MIN_LEN = 200  # justera vid behov (t.ex. 200 = ~1 sek vid 200 Hz)

short_files = []

for part_num, block_num, file in participant_files:
    try:
        df = pd.read_csv(file)

        # justera kolumnnamn om det behövs
        ppg = df["ppg"].dropna()
        eda = df["gsr"].dropna()

        length = min(len(ppg), len(eda))

        if length < MIN_LEN:
            short_files.append((part_num, block_num, file.name, length))

    except Exception as e:
        print(f"Error reading {file.name}: {e}")

# -------------------------
# Resultat
# -------------------------
print(f"\nAntal korta filer: {len(short_files)}\n")

for part, block, name, length in short_files:
    print(f"Part{part:02d} | Block {block:2d} | Length: {length:4d} → {name}")


Antal korta filer: 1

Part04 | Block  3 | Length:    0 → 3_gsr_ppg_.csv


In [12]:
df = pd.read_csv("../data/raw/Participants/Part4/by_block/3_gsr_ppg_.csv")
print(df.head())
print(df.info())

Empty DataFrame
Columns: [Timestamp, PythonTimestamp, accelx, accely, accelz, ppg, gsr]
Index: []
<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Timestamp        0 non-null      object
 1   PythonTimestamp  0 non-null      object
 2   accelx           0 non-null      object
 3   accely           0 non-null      object
 4   accelz           0 non-null      object
 5   ppg              0 non-null      object
 6   gsr              0 non-null      object
dtypes: object(7)
memory usage: 132.0+ bytes
None


In [24]:
all_features = []
start_time = time.time()

for i, (participant_num, block_num, file) in enumerate(
    tqdm(participant_files, desc="Processing files")
):
    participant_id = f"Part{participant_num}"

    print(
        f"\nProcessing {participant_id} | Block {block_num} "
        f"({i+1}/{len(participant_files)})"
    )

    try:
        df = pd.read_csv(file)
    except Exception as e:
        print(f"⛔ Could not read file: {file} | Error: {e}")
        continue

    # -------------------------
    # 🔥 Skip trasiga filer
    # -------------------------
    if df.empty:
        print(f"⛔ Skipping EMPTY file: {file}")
        continue

    # 🔥 Kontrollera kolumner (ppg + gsr)
    if "ppg" not in df.columns or "gsr" not in df.columns:
        print(f"⛔ Missing columns in: {file}")
        continue

    ppg = df["ppg"].dropna()
    gsr = df["gsr"].dropna()

    if len(ppg) == 0 or len(gsr) == 0:
        print(f"⛔ Skipping invalid signal: {file}")
        continue

    # -------------------------
    # Feature extraction
    # -------------------------
    try:
        features = extract_features_with_rr(
            df, participant_id
        )
    except Exception as e:
        print(f"⛔ Error processing {file}: {e}")
        continue

    # -------------------------
    # Metadata
    # -------------------------
    features["block"] = block_num
    features["participant_num"] = participant_num

    all_features.append(features)

    # -------------------------
    # Time estimation
    # -------------------------
    elapsed = time.time() - start_time
    avg_time = elapsed / (i + 1)
    remaining = avg_time * (len(participant_files) - (i + 1))

    print(
        f"Elapsed: {elapsed/60:.1f} min | "
        f"Remaining: {remaining/60:.1f} min"
    )

Processing files:   0%|          | 0/816 [00:00<?, ?it/s]


Processing Part1 | Block 1 (1/816)


Processing files:   0%|          | 1/816 [00:00<12:38,  1.08it/s]

Elapsed: 0.0 min | Remaining: 12.9 min

Processing Part1 | Block 2 (2/816)


Processing files:   0%|          | 3/816 [00:03<12:49,  1.06it/s]

...

Elapsed: 7.5 min | Remaining: 0.0 min

Processing Part48 | Block 28 (815/816)


Processing files: 100%|█████████▉| 815/816 [07:28<00:00,  2.79it/s]

Elapsed: 7.5 min | Remaining: 0.0 min

Processing Part48 | Block 33 (816/816)


Processing files: 100%|██████████| 816/816 [07:29<00:00,  1.82it/s]

Elapsed: 7.5 min | Remaining: 0.0 min


In [25]:
import numpy as np

features_df = pd.concat(all_features, ignore_index=True)

EPS = 1e-6

# -------------------------
# 🔥 EDA (GSR) transformation
# -------------------------
features_df["EDA_Tonic_log"] = np.log(
    features_df["EDA_Tonic"] - features_df["EDA_Tonic"].min() + EPS
)

features_df["EDA_Phasic_log"] = np.log(
    np.abs(features_df["EDA_Phasic"]) + EPS
)

# -------------------------
# 🔥 HRV clipping
# -------------------------
features_df["HRV_RMSSD"] = features_df["HRV_RMSSD"].clip(0, 200)
features_df["HRV_SDNN"]  = features_df["HRV_SDNN"].clip(0, 200)

# -------------------------
# 🔥 SCR (log-transform)
# -------------------------
features_df["SCR_Count"] = np.log1p(features_df["SCR_Count"])

# -------------------------
# 🔥 EDA clipping
# -------------------------
features_df["EDA_Tonic_log"]  = features_df["EDA_Tonic_log"].clip(6.0, 8.0)
features_df["EDA_Phasic_log"] = features_df["EDA_Phasic_log"].clip(-4.0, 2.0)

print("✅ Post-processing klar")
print(features_df.head())

✅ Post-processing klar
          HR   HRV_RMSSD    HRV_SDNN     RR_mean      RR_std  RR_min  RR_max  \
0  90.925119   99.890091  140.229538  667.954545  138.626863   340.0   960.0   
1  90.305141   99.890091  147.451156  675.444444  145.803605   340.0  1005.0   
2  91.902019   98.623336  155.224015  658.777778  153.489614   340.0  1005.0   
3  92.594164  101.033720  154.118657  654.130435  152.434249   340.0  1005.0   
4  93.213341   98.004279  152.809637  652.717391  151.139536   340.0  1005.0   

   RR_count  RR_diff_std  RR_valid   EDA_Tonic  EDA_Phasic  SCR_Count  \
0      44.0    99.715159       1.0  566.866924   16.358416        0.0   
1      45.0    99.715159       1.0  578.537855   13.859142        0.0   
2      45.0    98.176738       1.0  592.559782    9.136078        0.0   
3      46.0   101.002785       1.0  609.822569    2.700045        0.0   
4      46.0    97.671911       1.0  631.302652   -7.301003        0.0   

  participant  time_sec  block  participant_num  EDA_Toni

In [26]:
# -------------------------
# 🔍 Grundläggande kontroll
# -------------------------
print("\nShape:", features_df.shape)
print("\nMissing values:")
print(features_df.isna().sum())

print("\nBeskrivning:")
print(features_df.describe())

# -------------------------
# 🔍 Kontrollera extrema värden
# -------------------------
print("\nHRV max:", features_df["HRV_RMSSD"].max(), features_df["HRV_SDNN"].max())
print("EDA tonic range:", features_df["EDA_Tonic_log"].min(), features_df["EDA_Tonic_log"].max())
print("EDA phasic range:", features_df["EDA_Phasic_log"].min(), features_df["EDA_Phasic_log"].max())

# -------------------------
# 💾 Spara till CSV
# -------------------------
OUTPUT_PATH = Path("../data/processed/features_cleaned_by_block_with_rr.csv")

features_df.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Sparad till: {OUTPUT_PATH}")


Shape: (28506, 19)

Missing values:
HR                 0
HRV_RMSSD          0
HRV_SDNN           0
RR_mean            0
RR_std             0
RR_min             0
RR_max             0
RR_count           0
RR_diff_std        0
RR_valid           0
EDA_Tonic          0
EDA_Phasic         0
SCR_Count          0
participant        0
time_sec           0
block              0
participant_num    0
EDA_Tonic_log      0
EDA_Phasic_log     0
dtype: int64

Beskrivning:
                 HR     HRV_RMSSD      HRV_SDNN       RR_mean        RR_std  \
count  28506.000000  28506.000000  28506.000000  28506.000000  28506.000000   
mean      82.776742     52.925628     56.786369    739.761344     56.081729   
std       11.571709     21.240796     27.365166    102.334813     27.084777   
min       57.230703      8.206518     11.439727    424.000000     11.324752   
25%       74.789661     36.462458     37.463444    669.666667     37.035107   
50%       82.061150     52.363769     51.841265    732.195122  

In [27]:
# -------------------------
# Feature columns used for SOM
# -------------------------
FEATURE_COLS = [
    "HR",
    "HRV_RMSSD",
    "HRV_SDNN",
    "SCR_Count",
    "EDA_Tonic_log",
    "EDA_Phasic_log"
]

# -------------------------
# 🔥 Ta bort rader med saknade värden
# -------------------------
features_clean = features_df.dropna(subset=FEATURE_COLS).copy()

print("Original rows:", len(features_df))
print("After cleaning:", len(features_clean))
print("Removed rows:", len(features_df) - len(features_clean))

# -------------------------
# SOM input matrix
# -------------------------
X_features = features_clean[FEATURE_COLS]

# -------------------------
# Metadata (matchar exakt)
# -------------------------
metadata = features_clean[["participant_num", "block"]]

print("\n✅ SOM data redo")
print("Shape:", X_features.shape)

Original rows: 28506
After cleaning: 28506
Removed rows: 0

✅ SOM data redo
Shape: (28506, 6)


In [28]:
X_features.describe()

,HR,HRV_RMSSD,HRV_SDNN,SCR_Count,EDA_Tonic_log,EDA_Phasic_log
count,28506.000000,28506.000000,28506.000000,28506.000000,28506.000000,28506.000000
mean,82.776742,52.925628,56.786369,1.205428,6.909535,-1.943580
std,11.571709,21.240796,27.365166,0.605875,0.381713,1.559181
min,57.230703,8.206518,11.439727,0.000000,6.000000,-4.000000
25%,74.789661,36.462458,37.463444,0.693147,6.667797,-3.180727
50%,82.061150,52.363769,51.841265,1.386294,6.774604,-2.097272
75%,89.748962,69.531399,68.065954,1.609438,6.953382,-1.000797
max,141.504412,131.700418,200.000000,3.465736,8.000000,2.000000


In [29]:
OUTPUT_PATH = Path("../data/processed/features_cleaned_by_block_with_rr_SOM.csv")

X_features.to_csv(OUTPUT_PATH, index=False)

print(f"\n✅ Sparad till: {OUTPUT_PATH}")


✅ Sparad till: ..\data\processed\features_cleaned_by_block_with_rr_SOM.csv
